In [2]:
# Install OpenAI library
!pip install -q openai

# Imports
import base64
from openai import OpenAI
from google.colab import files

# Upload image
print("Upload an image...")
uploaded = files.upload()
image_path = list(uploaded.keys())[0]

# Convert image to base64
with open(image_path, "rb") as img:
    base64_image = base64.b64encode(img.read()).decode("utf-8")

# Initialize OpenAI client
client = OpenAI(api_key="")

# Ask a question about the image
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe what you see in this image."},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    }
                }
            ]
        }
    ]
)

# Print response
print("\nModel Response:")
print(response.choices[0].message.content)

Upload an image...


Saving D701A6CD-ACF0-466F-AA2D-FA24081E2790.PNG to D701A6CD-ACF0-466F-AA2D-FA24081E2790.PNG

Model Response:
The image features four heart-shaped boxes, each with a different color: blue, pink, purple, and white. Each box is open and displays a text screen inside, which reads, "You are this one in [my] love, and [...] me into baby[s]." The boxes are placed against a soft, light-colored background with small hearts and floral decor, creating a romantic ambiance. The design appears modern and playful, suggesting they might be used for gifting or expressing affection.


In [6]:
# Install dependencies
!pip install -q openai pandas pillow

# Imports
import base64
import json
import pandas as pd
from PIL import Image
from openai import OpenAI
from google.colab import files

# Initialize OpenAI
client = OpenAI(api_key="")

print("Upload prescription / bill / receipt images")
uploaded = files.upload()

rows = []

for image_name in uploaded.keys():

    # Convert image to base64
    with open(image_name, "rb") as img:
        base64_image = base64.b64encode(img.read()).decode("utf-8")

    prompt = """
    Extract structured data from this document.
    The document could be a medical prescription, bill, invoice, or receipt.

    Return JSON with fields like:
    {
      "document_type": "",
      "date": "",
      "vendor_or_hospital": "",
      "items": [
        {"name": "", "quantity": "", "price": ""}
      ],
      "total_amount": ""
    }
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type":"text","text":prompt},
                    {
                        "type":"image_url",
                        "image_url":{
                            "url":f"data:image/jpeg;base64,{base64_image}"
                        }
                    }
                ]
            }
        ],
        response_format={"type": "json_object"}
    )

    data = json.loads(response.choices[0].message.content)

    # Flatten items into rows
    for item in data.get("items", []):
        rows.append({
            "Document Type": data.get("document_type"),
            "Date": data.get("date"),
            "Vendor/Hospital": data.get("vendor_or_hospital"),
            "Item": item.get("name"),
            "Quantity": item.get("quantity"),
            "Price": item.get("price"),
            "Total Amount": data.get("total_amount")
        })

# Convert to table
df = pd.DataFrame(rows)

print("\nExtracted Table:")
display(df)

# Save to CSV
df.to_csv("extracted_documents.csv", index=False)

print("\nCSV saved as extracted_documents.csv")

Upload prescription / bill / receipt images


Saving ChatGPT Image Mar 12, 2026, 11_40_14 AM.png to ChatGPT Image Mar 12, 2026, 11_40_14 AM.png

Extracted Table:


,Document Type,Date,Vendor/Hospital,Item,Quantity,Price,Total Amount
0,Prescription & Pharmacy Bill,04/24/2024,General Medical Center,Metformin,60 tablets,$20.00,$139.85
1,Prescription & Pharmacy Bill,04/24/2024,General Medical Center,Lisinopril,30 tablets,$15.00,$139.85
2,Prescription & Pharmacy Bill,04/24/2024,General Medical Center,Cough Syrup,1 bottle,$12.00,$139.85
3,Prescription & Pharmacy Bill,04/24/2024,General Medical Center,Atorvastatin,30 tablets,$25.00,$139.85



CSV saved as extracted_documents.csv


In [7]:
!pip install openai pandas pillow

import base64
import pandas as pd
from openai import OpenAI
from google.colab import files

client = OpenAI(api_key="")

print("Upload files (audio, image, or document)")
uploaded = files.upload()

rows = []

for file in uploaded:

    if file.endswith(".jpg") or file.endswith(".png"):

        with open(file,"rb") as f:
            img = base64.b64encode(f.read()).decode()

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role":"user",
                    "content":[
                        {"type":"text","text":"Extract items, quantity, price from this document"},
                        {"type":"image_url","image_url":{"url":f"data:image/jpeg;base64,{img}"}}
                    ]
                }
            ]
        )

        text = response.choices[0].message.content

    elif file.endswith(".wav") or file.endswith(".mp3"):

        audio = open(file,"rb")

        transcript = client.audio.transcriptions.create(
            model="gpt-4o-mini-transcribe",
            file=audio
        )

        text = transcript.text

    else:

        text = open(file).read()

    rows.append({"content": text})

df = pd.DataFrame(rows)

print(df)
df.to_csv("processed_data.csv")

Upload files (audio, image, or document)


Saving 17732959987387185972gm733xd-voicemaker.in-speech.mp3 to 17732959987387185972gm733xd-voicemaker.in-speech.mp3
                                             content
0  Prescribed amoxicillin 500 milligrams twice da...
